In [ ]:
!nvidia-smi
%pip install torchmetrics[image] -q

In [ ]:

from __future__ import annotations
from typing import Callable, Iterable, Optional

import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.utils as vutils

from torch import Tensor
from torch.optim import Optimizer

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance

In [ ]:
SEED = 42 # For robust reporting, run multiple seeds: 42, 6, 11, 96, 22
random.seed(SEED) # Python's built-in RNG
np.random.seed(SEED) # NumPy's RNG
torch.manual_seed(SEED) # PyTorch's CPU RNG
torch.cuda.manual_seed_all(SEED) # PyTorch's CUDA RNG

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class HNAdam(Optimizer):
    def __init__(
        self,
        params: Iterable[Tensor],
        lr: float = 1e-3,
        betas: tuple[float, float] = (0.9, 0.99),
        eps: float = 1e-8,
        lambda_t0: Optional[float] = None,
    ) -> None:
        if params is None:
            raise ValueError("params cannot be None.")
        if lr <= 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if eps < 0.0:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if len(betas) != 2:
            raise ValueError("betas must be a tuple of two floats")

        beta1, beta2 = betas
        if not 0.0 <= beta1 < 1.0:
            raise ValueError(f"Invalid beta1 value: {beta1}")
        if not 0.0 <= beta2 < 1.0:
            raise ValueError(f"Invalid beta2 value: {beta2}")

        if lambda_t0 is None:
            lambda_t0 = random.uniform(2.0, 4.0)
        if not 2.0 <= lambda_t0 <= 4.0:
            raise ValueError(f"lambda_t0 must be in [2, 4], got {lambda_t0}")

        defaults = {
            "lr": lr,
            "betas": (beta1, beta2),
            "eps": eps,
            "lambda_t0": lambda_t0,
            "amsgrad": False,
        }
        super().__init__(params, defaults)

        if len(self.param_groups) == 0:
            raise ValueError("optimizer got an empty parameter list")

    @torch.no_grad()
    def step(self, closure: Optional[Callable[[], Tensor]] = None) -> Optional[Tensor]:
        loss: Optional[Tensor] = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr: float = group["lr"]
            beta1, beta2 = group["betas"]
            eps: float = group["eps"]
            lambda_t0: float = group["lambda_t0"]

            for param in group["params"]:
                if param.grad is None:
                    continue

                grad = param.grad
                if grad.is_sparse:
                    raise RuntimeError("HNAdam does not support sparse gradients")

                state = self.state[param]

                if len(state) == 0:
                    state["m"] = torch.zeros_like(param, memory_format=torch.preserve_format)
                    state["v"] = torch.zeros_like(param, memory_format=torch.preserve_format)
                    state["vhat"] = torch.zeros_like(param, memory_format=torch.preserve_format)

                m_prev: Tensor = state["m"]
                v_prev: Tensor = state["v"]
                vhat_prev: Tensor = state["vhat"]

                g_t = grad

                m_t = beta1 * m_prev + (1.0 - beta1) * g_t

                g_abs = g_t.abs()
                m_prev_norm = torch.linalg.vector_norm(m_prev)
                g_abs_norm = torch.linalg.vector_norm(g_abs)
                m_max = torch.maximum(m_prev_norm, g_abs_norm)

                zero = torch.zeros((), dtype=param.dtype, device=param.device)
                ratio = torch.where(m_max > 0.0, m_prev_norm / m_max, zero)
                lambda_t = torch.as_tensor(lambda_t0, dtype=param.dtype, device=param.device) - ratio

                v_t = beta2 * v_prev + (1.0 - beta2) * g_abs.pow(lambda_t)

                if bool((lambda_t < 2.0).item()):
                    group["amsgrad"] = True

                    vhat_t = torch.maximum(vhat_prev, v_t.abs())
                    state["vhat"] = vhat_t

                    denom = vhat_t.pow(1.0 / lambda_t) + eps
                else:
                    group["amsgrad"] = False

                    denom = v_t.pow(1.0 / lambda_t) + eps

                param.addcdiv_(m_t, denom, value=-lr)

                state["m"] = m_t
                state["v"] = v_t

        return loss

In [ ]:
# Inputs

# Image
IMAGE_SIZE = 64 # Spatial size of training images. All images will be resized to this size using a transformer.
NUM_CHANNELS = 3 # Number of channels in the training images. For color images this is 3.

# Model
NUM_FILTERS = 64 # Size of feature maps in the first layer of the generator and discriminator.
LATENT_DIM = 100 # Size of the latent vector (input to the generator).

# Training
BATCH_SIZE = 128 # The DCGAN paper uses a batch size of 128
EPOCHS = 5 # Set to 5 for quick testing
LR = 2e-4 # The DCGAN paper uses a learning rate of 0.0002
BETA1 = 0.5 # The DCGAN paper uses a beta1 of 0.5 for Adam optimizers
BETA2 = 0.999 
EPS = 1e-8
REAL_LABEL = 1.0
FAKE_LABEL = 0.0

# Define transform to apply to input images
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])  

# Load dataset
train_dataset = datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root="./data", train=False, transform=transform, download=True)

# Instantiate DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')

# Plot some training images
real_batch = next(iter(train_loader))
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Training Images")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], padding=2, normalize=True).cpu(), (1, 2, 0)))
plt.show()

In [ ]:
# Custom weights initialization called on netG and netD
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
# Generator Code
class Generator(nn.Module):
    def __init__(self, LATENT_DIM, NUM_FILTERS, NUM_CHANNELS):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(LATENT_DIM, NUM_FILTERS * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NUM_FILTERS * 8),
            nn.ReLU(True),
            # state size. (NUM_FILTERS*8) x 4 x 4
            nn.ConvTranspose2d(NUM_FILTERS * 8, NUM_FILTERS * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NUM_FILTERS * 4),
            nn.ReLU(True),
            # state size. (NUM_FILTERS*4) x 8 x 8
            nn.ConvTranspose2d(NUM_FILTERS * 4, NUM_FILTERS * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NUM_FILTERS * 2),
            nn.ReLU(True),
            # state size. (NUM_FILTERS*2) x 16 x 16
            nn.ConvTranspose2d(NUM_FILTERS * 2, NUM_FILTERS, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NUM_FILTERS),
            nn.ReLU(True),
            # state size. (NUM_FILTERS) x 32 x 32
            nn.ConvTranspose2d(NUM_FILTERS, NUM_CHANNELS, 4, 2, 1, bias=False),
            nn.Tanh()
            # state size. (NUM_CHANNELS) x 64 x 64
        )
    def forward(self, input):
        return self.main(input)

In [ ]:
# Create the generator
netG = Generator(LATENT_DIM, NUM_FILTERS, NUM_CHANNELS).to(device)

# Apply the weights_init function to randomly initialize all weights to mean=0, stdev=0.02.
netG.apply(weights_init)

# Print the model
print(netG)

In [ ]:
# Discriminator Code
class Discriminator(nn.Module):
    def __init__(self, NUM_CHANNELS, NUM_FILTERS):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # input is (NUM_CHANNELS) x 64 x 64
            nn.Conv2d(NUM_CHANNELS, NUM_FILTERS, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (NUM_FILTERS) x 32 x 32
            nn.Conv2d(NUM_FILTERS, NUM_FILTERS * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NUM_FILTERS * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (NUM_FILTERS*2) x 16 x 16
            nn.Conv2d(NUM_FILTERS * 2, NUM_FILTERS * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NUM_FILTERS * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (NUM_FILTERS*4) x 8 x 8
            nn.Conv2d(NUM_FILTERS * 4, NUM_FILTERS * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NUM_FILTERS * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (NUM_FILTERS*8) x 4 x 4
            nn.Conv2d(NUM_FILTERS * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, input):
        return self.main(input)

In [ ]:
# Create the Discriminator
netD = Discriminator(NUM_CHANNELS, NUM_FILTERS).to(device)

# Apply the weights_init function to randomly initialize all weights to mean=0, stdev=0.02.
netD.apply(weights_init)

# Print the model
print(netD)

In [ ]:
# Initialize the BCELoss function
criterion = nn.BCELoss()

# Fixed latent vectors used for consistent sample snapshots across runs
fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)

# Optimizers to compare
optimizer_names = ["HN_Adam", "Adam", "AMSGrad", "SGD"]

def build_optimizer_pair(optimizer_name, netD, netG):
    if optimizer_name == "HN_Adam":
        optimizerD = HNAdam(netD.parameters(), lr=LR, betas=(BETA1, BETA2), eps=EPS)
        optimizerG = HNAdam(netG.parameters(), lr=LR, betas=(BETA1, BETA2), eps=EPS)
    elif optimizer_name == "Adam":
        optimizerD = torch.optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, BETA2), eps=EPS)
        optimizerG = torch.optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, BETA2), eps=EPS)
    elif optimizer_name == "AMSGrad":
        optimizerD = torch.optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, BETA2), eps=EPS, amsgrad=True)
        optimizerG = torch.optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, BETA2), eps=EPS, amsgrad=True)
    elif optimizer_name == "SGD":
        optimizerD = torch.optim.SGD(netD.parameters(), lr=LR)
        optimizerG = torch.optim.SGD(netG.parameters(), lr=LR)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")

    return optimizerD, optimizerG

In [ ]:
# Training loop for optimizer comparison
comparison_results = {}

print("Starting optimizer comparison training loop...")
overall_start_time = time.perf_counter()

for optimizer_name in optimizer_names:
    print(f"\n=== Training with {optimizer_name} ===")

    # Reset model weights so each optimizer starts from the same initialization
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    netG = Generator(LATENT_DIM, NUM_FILTERS, NUM_CHANNELS).to(device)
    netG.apply(weights_init)

    netD = Discriminator(NUM_CHANNELS, NUM_FILTERS).to(device)
    netD.apply(weights_init)

    optimizerD, optimizerG = build_optimizer_pair(optimizer_name, netD, netG)

    # Use a fresh dataloader per run for clean iteration state
    run_loader = train_loader

    img_list = []
    G_losses = []
    D_losses = []
    iters = 0

    start_time = time.perf_counter()

    # For each epoch
    for epoch in range(EPOCHS):
        # For each batch in the train_loader
        for i, data in enumerate(run_loader, 0):

            # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
            netD.zero_grad()

            # Format batch
            real_cpu = data[0].to(device)
            b_size = real_cpu.size(0)
            label = torch.full((b_size,), REAL_LABEL, dtype=torch.float, device=device)

            # Forward pass real batch through D
            output = netD(real_cpu).view(-1)

            # Calculate loss on all-real batch
            errD_real = criterion(output, label)

            # Calculate gradients for D in backward pass
            errD_real.backward()
            D_x = output.mean().item()

            # Train with all-fake batch
            noise = torch.randn(b_size, LATENT_DIM, 1, 1, device=device)

            # Generate fake image batch with G
            fake = netG(noise)
            label.fill_(FAKE_LABEL)

            # Classify all fake batch with D
            output = netD(fake.detach()).view(-1)

            # Calculate D's loss on the all-fake batch
            errD_fake = criterion(output, label)

            # Calculate the gradients for this batch, accumulated with previous gradients
            errD_fake.backward()
            D_G_z1 = output.mean().item()

            # Compute error of D as sum over the fake and the real batches
            errD = errD_real + errD_fake

            # Update D
            optimizerD.step()

            # (2) Update G network: maximize log(D(G(z)))
            netG.zero_grad()
            label.fill_(REAL_LABEL)

            # Since we just updated D, perform another forward pass of all-fake batch through D
            output = netD(fake).view(-1)

            # Calculate G's loss based on this output
            errG = criterion(output, label)

            # Calculate gradients for G
            errG.backward()
            D_G_z2 = output.mean().item()

            # Update G
            optimizerG.step()

            # Output training stats
            if i % 50 == 0:
                print(
                    f"[{optimizer_name}] [{epoch}/{EPOCHS}][{i}/{len(run_loader)}]\t"
                    f"Loss_D: {errD.item():.4f}\tLoss_G: {errG.item():.4f}\t"
                    f"D(x): {D_x:.4f}\tD(G(z)): {D_G_z1:.4f} / {D_G_z2:.4f}"
                )

            # Save losses for plotting later
            G_losses.append(errG.item())
            D_losses.append(errD.item())

            # Save G's output on fixed_noise
            if (iters % 500 == 0) or ((epoch == EPOCHS - 1) and (i == len(run_loader) - 1)):
                with torch.no_grad():
                    fake_preview = netG(fixed_noise).detach().cpu()
                img_list.append(vutils.make_grid(fake_preview, padding=2, normalize=True))

            iters += 1

    training_time_seconds = time.perf_counter() - start_time
    print(f"{optimizer_name} training completed in {training_time_seconds:.2f} seconds.")

    comparison_results[optimizer_name] = {
        "netG": netG,
        "netD": netD,
        "img_list": img_list,
        "G_losses": G_losses,
        "D_losses": D_losses,
        "training_time_seconds": training_time_seconds,
    }

total_training_time = time.perf_counter() - overall_start_time
print(f"\nAll optimizer runs completed in {total_training_time:.2f} seconds.")

In [ ]:
# Loss versus training iterations for each optimizer
for optimizer_name, run_data in comparison_results.items():
    plt.figure(figsize=(10, 4))
    plt.title(f"{optimizer_name}: Generator and Discriminator Loss During Training")
    plt.plot(run_data["G_losses"], label="Generator Loss", color="tab:blue")
    plt.plot(run_data["D_losses"], label="Discriminator Loss", color="tab:orange")
    plt.xlabel("Iterations")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# SSIM and FID evaluation
num_eval_samples = 2048
table_rows = []

for optimizer_name, run_data in comparison_results.items():
    netG_eval = run_data["netG"]
    netG_eval.eval()

    samples_seen = 0
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(device)

    eval_loader = test_loader

    with torch.no_grad():
        for real_batch, _ in eval_loader:
            real_batch = real_batch.to(device)
            batch_size = real_batch.size(0)

            noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
            fake_batch = netG_eval(noise)

            if real_batch.shape[-2:] != fake_batch.shape[-2:]:
                real_batch = F.interpolate(real_batch, size=fake_batch.shape[-2:], mode="bilinear", align_corners=False)

            real_01 = ((real_batch + 1.0) / 2.0).clamp(0.0, 1.0)
            fake_01 = ((fake_batch + 1.0) / 2.0).clamp(0.0, 1.0)

            ssim_metric.update(fake_01, real_01)

            real_fid = (real_01 * 255.0).to(torch.uint8)
            fake_fid = (fake_01 * 255.0).to(torch.uint8)
            fid_metric.update(real_fid, real=True)
            fid_metric.update(fake_fid, real=False)

            samples_seen += batch_size
            if samples_seen >= num_eval_samples:
                break

    ssim_value = float(ssim_metric.compute().item())
    fid_value = float(fid_metric.compute().item())

    run_data["SSIM"] = ssim_value
    run_data["FID"] = fid_value

    table_rows.append(
        {
            "Optimizer": optimizer_name,
            "Training Time (s)": run_data["training_time_seconds"],
            "SSIM": ssim_value,
            "FID": fid_value,
        }
    )

table_1 = pd.DataFrame(table_rows).set_index("Optimizer")
table_1.index.name = "Table 1"
display(table_1)

In [ ]:
# Real images vs fake images for each optimizer
real_batch = next(iter(train_loader))

num_cols = 1 + len(comparison_results)
plt.figure(figsize=(4 * num_cols, 4))

# Plot real images
plt.subplot(1, num_cols, 1)
plt.axis("off")
plt.title("Real Images")
plt.imshow(
    np.transpose(
        vutils.make_grid(real_batch[0].to(device)[:64], padding=5, normalize=True).cpu(),
        (1, 2, 0),
    )
)

# Plot latest fake images from each optimizer run
for idx, (optimizer_name, run_data) in enumerate(comparison_results.items(), start=2):
    plt.subplot(1, num_cols, idx)
    plt.axis("off")
    plt.title(f"Fake ({optimizer_name})")
    plt.imshow(np.transpose(run_data["img_list"][-1], (1, 2, 0)))

plt.tight_layout()
plt.show()